# Week 5 — Multi-Receiver Spatial Consistency

This notebook compares two static receivers with known separation.
Under common-mode spoofing, both receivers drift together.

Your task is to detect the attack by monitoring the **relative baseline** between the receivers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../datasets/week5_multi_receiver.csv")
df.head()

## Task 1: Pivot the table so both receivers appear on the same row

In [ ]:
wide = df.pivot(index="t_s", columns="receiver_id", values=["true_x_m","true_y_m","meas_x_m","meas_y_m","spoof_active"])
wide.head()

## Task 2: Compute the measured baseline vector and its error

In [ ]:
bx = wide["meas_x_m"]["B"] - wide["meas_x_m"]["A"]
by = wide["meas_y_m"]["B"] - wide["meas_y_m"]["A"]

true_bx = wide["true_x_m"]["B"] - wide["true_x_m"]["A"]
true_by = wide["true_y_m"]["B"] - wide["true_y_m"]["A"]

baseline_error = np.sqrt((bx - true_bx)**2 + (by - true_by)**2)

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(wide.index, baseline_error)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Baseline error [m]")
ax.set_title("Spatial consistency check")
plt.show()

## Task 3: Create a detector based on baseline error

In [ ]:
threshold = 4.0  # TODO: tune
detected = (baseline_error > threshold).astype(int)
truth = wide["spoof_active"]["A"].astype(int)

fig, ax = plt.subplots(figsize=(10,4))
ax.plot(wide.index, truth, label="truth")
ax.plot(wide.index, detected, label="detected")
ax.legend()
ax.set_xlabel("Time [s]")
plt.show()

## Stretch task
Compare this detector against a single-receiver detector based only on position drift, and discuss why the two-receiver baseline can reveal inconsistencies that a single drifting receiver cannot.